# Week 10: MLOps, System Evaluation & Explainable AI
## Notebook 09: Feature Importance: Permutation vs Tree-Based MDI

### 1. Conceptual Intuition & Theoretical Background
Understanding which features drive model decisions is critical for compliance and debugging.

**1. Mean Decrease in Impurity (MDI / Gini Importance):**
Calculates the total impurity reduction brought by a feature across all trees:
$$\text{MDI}(X_j) = \frac{1}{T} \sum_{t=1}^{T} \sum_{n \in \text{splits}(t, j)} p(n) \Delta I(n)$$
- **Severe Flaws:** Heavily biased toward **high-cardinality features** (e.g. unique user IDs or timestamps) and evaluated on training data, causing overfitting to noise.

**2. Permutation Feature Importance (Breiman, 2001):**
Evaluates model score on a held-out validation set, randomly shuffles feature $X_j$ to break its relationship with the target, and measures performance drop:
$$\text{PFI}(X_j) = \text{Metric}(X_{\text{val}}, y) - \text{Metric}(X_{\text{val}}^{(\text{shuffled } j)}, y)$$
- **Advantage:** Evaluates generalization on unseen validation data and does not suffer from cardinality bias.

---

### 2. Scratch Python Implementation: Permutation Importance Engine

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

def permutation_importance_scratch(model, X_val, y_val, metric_fn, n_repeats=5):
    """
    Computes Permutation Feature Importance on held-out validation data.
    """
    baseline_score = metric_fn(y_val, model.predict(X_val))
    importances = {}
    
    for col in X_val.columns:
        drops = []
        X_permuted = X_val.copy()
        
        for _ in range(n_repeats):
            # Shuffle single column
            X_permuted[col] = np.random.permutation(X_permuted[col].values)
            perm_score = metric_fn(y_val, model.predict(X_permuted))
            drops.append(baseline_score - perm_score)
            
        importances[col] = (np.mean(drops), np.std(drops))
        
    return importances

# Demonstration of MDI Cardinality Bias
np.random.seed(42)
n_samples = 500

# True informative feature
informative_feat = np.random.normal(0, 1, n_samples)
target = (informative_feat > 0).astype(int)

# Uninformative High-Cardinality Random ID feature
high_cardinality_noise = np.random.randint(0, 100000, n_samples)

X_train = pd.DataFrame({'informative': informative_feat, 'noise_id': high_cardinality_noise})
X_val = pd.DataFrame({
    'informative': np.random.normal(0, 1, 200),
    'noise_id': np.random.randint(0, 100000, 200)
})
y_val = (X_val['informative'] > 0).astype(int)

rf = RandomForestClassifier(n_estimators=30, random_state=42).fit(X_train, target)

# Accuracy metric
def accuracy_metric(y_true, y_pred):
    return np.mean(y_true == y_pred)

pfi_scores = permutation_importance_scratch(rf, X_val, y_val, accuracy_metric)

print("--- Feature Importance Diagnostics ---")
print("1. Tree-Based MDI Importance (Impurity Reduction on Train):")
for col, mdi in zip(X_train.columns, rf.feature_importances_):
    print(f"  {col}: {mdi:.4f}")
    
print("\n2. Permutation Importance (Performance Drop on Validation):")
for col, (mean_drop, std_drop) in pfi_scores.items():
    print(f"  {col}: {mean_drop:.4f} (+/- {std_drop:.4f})")
print("\nMDI falsely flags noise_id as important! Permutation importance correctly identifies it as 0.")

### 3. Step-by-Step Code Walkthrough
1. **MDI Cardinality Trap:** Shows how tree-based impurity falsely favors high-cardinality noise.
2. **Column Shuffling:** Breaks the statistical association with the target while maintaining column marginals.
3. **Generalization Drop:** Evaluates performance decline on unseen validation data.

---

### 4. Industry Applications & Trade-offs
- **Regulatory Audits & Feature Pruning:** Discarding redundant features without risk of cardinality bias.
- **Trade-off:** Permuting correlated features can evaluate the model on unrealistic synthetic feature combinations.

---

### 5. Top Interview Questions & Answers
**Q1: Why is default Random Forest feature importance (MDI) misleading for numerical or high-cardinality features?**
> **Answer:** Continuous and high-cardinality features provide many split opportunities, allowing trees to artificially lower node impurity during training even when features are uninformative noise.

**Q2: What is the downside of Permutation Feature Importance when features are collinear?**
> **Answer:** Shuffling one collinear feature still leaves its correlated counterparts intact, underestimating the importance of both features. Grouped permutation importance resolves this by shuffling correlated clusters together.